# Домашнє завдання: Рекомендаційні системи на реальних даних (Goodbooks-10k)

У цьому завданні Ви реалізуєте сучасні (advanced) архітектури рекомендаційних систем із фінального блоку лекції — але вже **не на іграшкових даних, а на реальному датасеті книжкових рейтингів Goodbooks-10k** (десятки тисяч користувачів, тисячі книг, мільйони оцінок).

Це дасть Вам змогу побачити, як підходи поводяться, коли даних справді багато: чому контентних ознак буває замало, як працює retrieval на тисячах елементів, і чому офлайн-метрики на кшталт Recall@K не такі високі, як хотілося б.

**Архітектури, які Ви зберете:** Vector Space Model, Two-Tower, Concat-based ranking (NCF) та двоетапний пайплайн Retrieval → Ranking.

**Стек:** `numpy`, `pandas`, `scikit-learn`, `torch`. GPU не обов'язковий, але з ним тренування буде швидшим (у Colab: *Runtime → Change runtime type → GPU*).

---

## Про датасет

[Goodbooks-10k](https://www.kaggle.com/datasets/zygmunt/goodbooks-10k) — це ~6 млн оцінок 10 000 найпопулярніших книг від 53 424 користувачів. Складається з кількох файлів:

- `ratings.csv` — оцінки: `user_id, book_id, rating` (1–5);
- `books.csv` — метадані книг: `book_id, goodreads_book_id, authors, title, average_rating, ...`;
- `book_tags.csv` — теги/полиці, які користувачі вішали на книги: `goodreads_book_id, tag_id, count`;
- `tags.csv` — розшифровка тегів: `tag_id, tag_name`.

**Важливий нюанс:** на відміну від навчального прикладу, тут **немає готових жанрів**. Жанри доведеться сконструювати самостійно з користувацьких тегів — а це шумні дані (юзери можуть зазначати що завгодно). Це реалістична задача feature engineering, і ми її розберемо в підготовчій частині.

Ще один нюанс із реальних даних: `book_tags.csv` посилається на `goodreads_book_id`, а `ratings.csv` — на `book_id`. Щоб їх поєднати, потрібен джойн через `books.csv`.


## Крок 0. Завантаження даних

Є три способи дістати дані — оберіть будь-який.

**Спосіб A — Kaggle API (рекомендований).** Завантаження з Kaggle API. Зручно, бо декілька файлів і вони завантажаться всі самостійно. Для цього способу завантажте свій `kaggle.json` (Kaggle → Account → Create New API Token), потім виконайте:
```python
from google.colab import files; files.upload()   # оберіть kaggle.json
```
і розкоментуйте відповідний блок нижче.

**Спосіб B — ручне завантаження.** Завантажте архів з посилання на датасет вище з Kaggle, розпакуйте і покладіть `ratings.csv`, `books.csv`, `book_tags.csv`, `tags.csv` поруч із ноутбуком (або через панель Files у Colab).

**Спосіб C — GitHub-дзеркало (фолбек).** Оригінальний автор виклав файли і на GitHub — код нижче підхопить їх автоматично, якщо локально файлів немає.


In [ ]:
# (Спосіб A) Kaggle API — розкоментуйте, якщо завантажили kaggle.json
# !pip -q install kaggle
# import os, shutil
# os.makedirs("/root/.kaggle", exist_ok=True)
# shutil.move("kaggle.json", "/root/.kaggle/kaggle.json"); os.chmod("/root/.kaggle/kaggle.json", 0o600)
# !kaggle datasets download -d zygmunt/goodbooks-10k --unzip -p .

In [1]:
import os
import numpy as np
import pandas as pd

GITHUB = "https://raw.githubusercontent.com/zygmuntz/goodbooks-10k/master"
FILES = ["ratings.csv", "books.csv", "book_tags.csv", "tags.csv"]

def load(fname):
    """Спочатку шукаємо файл локально, інакше тягнемо з GitHub-дзеркала."""
    if os.path.exists(fname):
        return pd.read_csv(fname)
    print(f"{fname} не знайдено локально — завантажую з GitHub...")
    return pd.read_csv(f"{GITHUB}/{fname}")

ratings = load("ratings.csv")
books = load("books.csv")
book_tags = load("book_tags.csv")
tags = load("tags.csv")

print("ratings:", ratings.shape)
print("books:  ", books.shape)
print("book_tags:", book_tags.shape, "| tags:", tags.shape)
books[["book_id", "authors", "title", "average_rating"]].head()

ratings.csv не знайдено локально — завантажую з GitHub...
books.csv не знайдено локально — завантажую з GitHub...
book_tags.csv не знайдено локально — завантажую з GitHub...
tags.csv не знайдено локально — завантажую з GitHub...
ratings: (5976479, 3)
books:   (10000, 23)
book_tags: (999912, 3) | tags: (34252, 2)


,book_id,authors,title,average_rating
0,1,Suzanne Collins,"The Hunger Games (The Hunger Games, #1)",4.34
1,2,"J.K. Rowling, Mary GrandPré",Harry Potter and the Sorcerer's Stone (Harry P...,4.44
2,3,Stephenie Meyer,"Twilight (Twilight, #1)",3.57
3,4,Harper Lee,To Kill a Mockingbird,4.25
4,5,F. Scott Fitzgerald,The Great Gatsby,3.89


## Крок 1. Інженерія жанрів із тегів (feature engineering)

Жанрів у датасеті немає, але є користувацькі теги. Виберемо набір канонічних жанрів і для кожної книги позначимо, які з них їй приписали користувачі. Так ми отримаємо **бінарну матрицю book × genre** — це й будуть контентні ознаки айтемів (аналог `movie_feats_df` із лекції, але здобутий з реальних шумних даних).


In [2]:
# Канонічні жанри, які шукаємо серед тегів
GENRES = ["fantasy", "romance", "mystery", "thriller", "horror", "historical",
          "science-fiction", "young-adult", "nonfiction", "classics",
          "contemporary", "crime"]

# tag_name -> tag_id
name_to_tagid = dict(zip(tags["tag_name"], tags["tag_id"]))
genre_tag_ids = {g: name_to_tagid[g] for g in GENRES if g in name_to_tagid}

# book_tags використовує goodreads_book_id -> мапимо у book_id через books.csv
gid_to_bid = dict(zip(books["goodreads_book_id"], books["book_id"]))
tagid_to_genre = {tid: g for g, tid in genre_tag_ids.items()}

bt = book_tags[book_tags["tag_id"].isin(genre_tag_ids.values())].copy()
bt["book_id"] = bt["goodreads_book_id"].map(gid_to_bid)
bt = bt.dropna(subset=["book_id"])
bt["genre"] = bt["tag_id"].map(tagid_to_genre)

# бінарна матриця book × genre (жанр присутній, якщо користувачі його тегали)
genre_matrix = (
    bt.pivot_table(index="book_id", columns="genre", values="count", aggfunc="sum", fill_value=0)
      .reindex(columns=GENRES, fill_value=0) > 0
).astype(int)

print("Книг із хоча б одним жанром:", (genre_matrix.sum(axis=1) > 0).sum(), "/", len(books))
print("\nРозподіл жанрів:")
print(genre_matrix.sum().sort_values(ascending=False))
genre_matrix.head()

Книг із хоча б одним жанром: 9954 / 10000

Розподіл жанрів:
genre
contemporary       5287
fantasy            4259
romance            4251
mystery            3686
young-adult        3630
classics           2785
historical         2544
thriller           2522
science-fiction    2222
crime              2083
nonfiction         1833
horror             1372
dtype: int64


genre,fantasy,romance,mystery,thriller,horror,historical,science-fiction,young-adult,nonfiction,classics,contemporary,crime
book_id,,,,,,,,,,,,
1,1,1,0,1,0,0,1,1,0,0,1,0
2,1,0,1,0,0,0,0,1,0,1,1,0
3,1,0,0,0,1,0,1,1,0,0,1,0
4,0,0,1,0,0,1,0,1,0,1,1,1
5,0,1,0,0,0,1,0,1,0,1,0,0


## Крок 2. Підвибірка під Colab

6 млн рейтингів — забагато для навчального ноутбука на CPU. Візьмемо **топ-N найпопулярніших книг** і **активних користувачів** (хто поставив ≥ 20 оцінок), а тоді обмежимо число користувачів. Так зберігається щільність взаємодій, а тренування лишається швидким.

> Якщо у Вас GPU або багато часу — сміливо збільшуйте `TOP_BOOKS` та `N_USERS`.


In [3]:
TOP_BOOKS = 1500       # скільки найпопулярніших книг лишити
MIN_USER_RATINGS = 20  # мінімум оцінок на користувача
N_USERS = 2000         # скільки користувачів узяти у підвибірку
LIKE_THRESHOLD = 4     # rating >= 4 вважаємо "лайком" (позитивна взаємодія)

rng = np.random.RandomState(42)

top_books = ratings["book_id"].value_counts().head(TOP_BOOKS).index
r = ratings[ratings["book_id"].isin(top_books)]
active = r["user_id"].value_counts()
r = r[r["user_id"].isin(active[active >= MIN_USER_RATINGS].index)]
sample_users = rng.choice(r["user_id"].unique(), size=min(N_USERS, r["user_id"].nunique()), replace=False)
r = r[r["user_id"].isin(sample_users)].copy()

# лишаємо тільки книги, для яких є жанрові ознаки
r = r[r["book_id"].isin(genre_matrix.index)].copy()

items = sorted(r["book_id"].unique())
users = sorted(r["user_id"].unique())
genre_matrix = genre_matrix.reindex(items).fillna(0).astype(int)

print(f"Взаємодій: {len(r):,} | користувачів: {len(users):,} | книг: {len(items):,}")
print(f"Щільність: {len(r) / (len(users) * len(items)):.4f}")

Взаємодій: 140,934 | користувачів: 2,000 | книг: 1,496
Щільність: 0.0471


In [4]:
import torch
import torch.nn as nn

torch.manual_seed(42)

user_to_idx = {u: i for i, u in enumerate(users)}
item_to_idx = {b: i for i, b in enumerate(items)}
title_of = dict(zip(books["book_id"], books["title"]))

#item_feats = torch.tensor(genre_matrix.values, dtype=torch.float32)  # (M, n_genres)
item_feats = torch.tensor(
    genre_matrix.loc[items].values,
    dtype=torch.float32
)

M = len(items)
n_genres = item_feats.shape[1]

# train/val split по взаємодіях
r = r.sample(frac=1, random_state=42).reset_index(drop=True)
n_val = int(len(r) * 0.2)
val_df = r.iloc[:n_val]
train_df = r.iloc[n_val:]

# позитивні пари (лайки) у train
train_pos = train_df[train_df["rating"] >= LIKE_THRESHOLD]
pos_u = torch.tensor([user_to_idx[u] for u in train_pos["user_id"]])
pos_i = torch.tensor([item_to_idx[b] for b in train_pos["book_id"]])

# що користувач уже бачив (щоб не рекомендувати повторно і не семплити як негатив)
from collections import defaultdict
seen_by_user = defaultdict(set)
for u, b in zip(train_df["user_id"], train_df["book_id"]):
    seen_by_user[user_to_idx[u]].add(item_to_idx[b])

# val-лайки для оцінки якості
val_pos = defaultdict(set)
for row in val_df.itertuples():
    if row.rating >= LIKE_THRESHOLD:
        val_pos[user_to_idx[row.user_id]].add(item_to_idx[row.book_id])

print(f"Позитивних пар у train: {len(pos_u):,} | користувачів з val-лайками: {len(val_pos):,}")

Позитивних пар у train: 77,070 | користувачів з val-лайками: 1,991


## Крок 3. Метрика оцінки якості рангування

В лекції ми з вами для оцінки якості використовували **RMSE**. Це валідний варіант, коли треба швидко оцінити якість рек. моделі, але спрощений. RMSE показує, наскільки точно модель передбачає оцінку, яку користувач поставить елементу.

В реальних системах нас ще цікавить **якість ранжування** — наскільки релевантні елементи потрапили в топ списку, який ми реально показуємо користувачу. Для цього використовують ранжувальні метрики: **Precision@K**, **Recall@K**, **NDCG**, **MAP**, **MRR**.

Детальніше можна познайомитись з цими мериками тут:
- огляд метрик для рекомендаційних систем: https://www.evidentlyai.com/ranking-metrics/evaluating-recommender-systems
- Precision та Recall at K: https://www.evidentlyai.com/ranking-metrics/precision-recall-at-k

Нижче давайте реалізуємо функцію `recall_at_k` і будемо оцінювати нею всі наші моделі.

![](https://cdn.prod.website-files.com/660ef16a9e0687d9cc27474a/662c4327f27ee08d3e4d4b2e_6577812c4d677925f1ab5f84_precision_recall_k9.png)

![](https://cdn.prod.website-files.com/660ef16a9e0687d9cc27474a/662c4327f27ee08d3e4d4b47_657781b1f9c868e0cda088f6_precision_recall_k11.png)

**Як працює `recall_at_k`:**

1. Для кожного користувача ми беремо його реальні вподобання з валідаційної вибірки (`val_pos` — книги, які він оцінив на ≥ 4), просимо модель оцінити всі книги й відбираємо топ-K рекомендацій. Перед цим прибираємо книги, які користувач уже бачив у train (щоб не рекомендувати відоме).

2. Далі рахуємо, скільки книг із топ-K справді потрапили в його вподобання (`hits`), і ділимо на загальну кількість релевантних книг (обмежену K, бо більше за K у топ і не влізе).

3. Усереднюємо по всіх користувачах — і отримуємо одне число від 0 до 1: **яку частку того, що користувачу реально сподобалось, модель змогла підняти в топ-K.**

In [5]:
def recall_at_k(score_fn, k=10):
    """Частка val-лайків, що потрапили у топ-k рекомендацій (усереднена по користувачах).
    score_fn(user_idx_tensor) -> матриця оцінок (n_users, M)."""
    eval_users = list(val_pos.keys())
    hits, total = 0, 0
    with torch.no_grad():
        scores = score_fn(torch.tensor(eval_users))  # (len(eval_users), M)
        for row, u in enumerate(eval_users):
            s = scores[row].clone()
            for i in seen_by_user[u]:
                s[i] = -1e9  # прибираємо вже побачене
            topk = torch.topk(s, k).indices.tolist()
            truth = val_pos[u]
            hits += len(set(topk) & truth)
            total += min(len(truth), k)
    return hits / max(total, 1)

---
## Завдання 1. Vector Space Model (векторний підхід)

Перетворимо і книги, і користувачів на вектори в спільному просторі та шукатимемо рекомендації через cosine similarity. Роль ембединга книги відіграє її **нормалізований вектор жанрів** (пояснення про нормалізацію - нижче), а вектор користувача збираємо як **average pooling** ембедингів книг, які він уподобав.

**Що зробити:**

1. Побудуйте `item_emb` — матрицю L2-нормалізованих жанрових векторів усіх книг.
2. Реалізуйте функцію `user_vector(user_idx)` — зважене (за оцінкою) середнє ембедингів уподобаних книг користувача.
3. Реалізуйте функцію `vsm_scores(user_idxs)` — оцінки (cosine) усіх книг для набору користувачів, та порахуйте `recall_at_k`.
4. Покажіть топ-5 рекомендацій для одного користувача (з назвами книг).

**Довідка:**

L2-нормалізація — це ділення вектора на його довжину (L2-норму), щоб отримати вектор тієї ж напрямленості, але одиничної довжини.

Норма рахується як корінь із суми квадратів компонент:

$$\|v\|_2 = \sqrt{(v_1^2 + v_2^2 + \dots + v_n^2)}$$

а сам нормалізований вектор — це
$$\hat{v} = \frac{v}{\|v\|_2}$$

Навіщо це в рекомендаційних системах: після нормалізації **косинусна подібність зводиться до простого скалярного добутку**. Бо $\cos(a, b) = \frac{a \cdot b}{\|a\|\|b\|}$, і якщо обидва вектори вже одиничної довжини, знаменник = 1, тож $\cos(a,b) = a \cdot b$. Це і швидше, і прибирає вплив «довжини» вектора — порівнюється лише напрямок (тобто склад жанрів/смаків), а не те, скільки книг користувач оцінив.

*Приклад:*

Вектор `[3, 4]` має довжину $\sqrt{(9+16)}=5$, після нормалізації стає `[0.6, 0.8]` — напрямок той самий, довжина 1.


### 1. Побудова item_emb

In [6]:
# жанрова матриця у правильному порядку items
genre_matrix_np = genre_matrix.loc[items].values.astype(np.float32)

def l2_normalize(matrix):
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    norms[norms == 0] = 1
    return matrix / norms

item_emb = torch.tensor(l2_normalize(genre_matrix_np), dtype=torch.float32)  # (M, n_genres)

### 2. Побудова матриці лайків (user × item)

In [7]:
user_item_likes = torch.zeros((len(users), len(items)), dtype=torch.float32)

for row in train_df.itertuples():
    if row.rating >= LIKE_THRESHOLD:
        u = user_to_idx[row.user_id]
        i = item_to_idx[row.book_id]
        user_item_likes[u, i] = row.rating  # вага = оцінка

### 3. Визначення user_vector

In [8]:
def user_vector(u_idx, item_emb, user_item_likes):
    likes = user_item_likes[u_idx]  # (M,)
    liked_items = likes > 0

    if liked_items.sum() == 0:
        return torch.zeros(item_emb.shape[1])

    weights = likes[liked_items]                     # (L,)
    vectors = item_emb[liked_items]                  # (L, n_genres)

    weighted = (vectors.T * weights).T               # (L, n_genres)
    avg = weighted.mean(dim=0)

    norm = torch.norm(avg)
    return avg / norm if norm > 0 else avg

### 4. Визначення vsm_scores

In [9]:
def vsm_scores(user_idxs):
    user_vecs = torch.stack([user_vector(u, item_emb, user_item_likes) for u in user_idxs])
    return user_vecs @ item_emb.T   # (n_users, M)

In [10]:
recall_vsm = recall_at_k(vsm_scores, k=10)
print("Recall@10 (VSM):", recall_vsm)

Recall@10 (VSM): 0.05211705706849489


**Питання:** Recall@10 у векторного підходу досить низький. Чому?


**Recall@10 низький, бо:**
- жанрові ознаки надто грубі
- вектори книг дуже схожі
- профіль користувача усереднений
- VSM не враховує негативи
- Goodreads - шумний датасет
- Recall@10 - сувора метрика
Модель працює правильно - просто жанрова інформація слабка.

### 5. Топ‑5 рекомендацій

In [11]:
def top_k_recommendations(user_id, k=5):
    u_idx = user_to_idx[user_id]
    scores = vsm_scores([u_idx])[0].clone()

    # прибираємо вже бачені
    for i in seen_by_user[u_idx]:
        scores[i] = -1e9

    top_items = torch.topk(scores, k).indices.tolist()
    return [(title_of[items[i]], float(scores[i])) for i in top_items]

In [12]:
example_user = users[0]
for title, score in top_k_recommendations(example_user):
    print(f"{title} — {score:.3f}")

Stardust — 0.885
The Clan of the Cave Bear (Earth's Children, #1) — 0.885
The Alchemist — 0.872
Inkheart (Inkworld, #1) — 0.868
Sophie's World — 0.865


---
## Завдання 2. Two-Tower архітектура

У Завданні 1 вектор користувача рахувався «вручну». Two-Tower натомість **навчає дві окремі башти**: User Tower (з ембединга user_id) та Item Tower (з жанрових ознак). Мережа зводить вектори уподобаних пар близько, а випадкових — далеко. Перевага: вектори книг рахуються один раз і кладуться в індекс (наприклад, FAISS) для швидкого retrieval — рахувати в реальному часі треба лише вектор користувача. Це **late fusion**.

**Що зробити:**

1. Реалізуйте `TwoTower` (user_tower через `nn.Embedding`, item_tower зі жанрових ознак), виходи L2-нормалізуйте.
2. Навчіть на лайках як позитивах і **negative sampling з усього корпусу** (як у пейпері від YouTube) з `BCEWithLogitsLoss` - він є реалізований в PyTorch.
3. Порахуйте `recall_at_k` через попередньо обчислені вектори книг і покажіть приклад рекомендацій.

> **Підказка.** Множте логіти на «температуру» (\~10), бо скалярний добуток нормалізованих векторів лежить у [-1, 1].
> Множення на температуру (\~10) розтягує діапазон логітів до [-10, 10], і тоді сигмоїда може видавати по-справжньому впевнені ймовірності (близькі до 0 і 1). Це дає лосу нормальний градієнт і модель навчається.


### 1. Ідея Two‑Tower
- User Tower: nn.Embedding(n_users, d)
- Item Tower: MLP над жанровими ознаками
- Обидва вектори → L2-нормалізація
- Схожість = dot(user_vec, item_vec)
- Логіти множимо на temperature=10
- Лосс: BCEWithLogitsLoss
- Negative sampling: випадкові книги

### 2. Реалізація TwoTower

In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class TwoTower(nn.Module):
    def __init__(self, n_users, n_genres, d=64, temperature=10.0):
        super().__init__()
        self.temperature = temperature

        self.user_emb = nn.Embedding(n_users, d)

        self.item_mlp = nn.Sequential(
            nn.Linear(n_genres, 128),
            nn.ReLU(),
            nn.Linear(128, d)
        )

    def forward(self, user_idx, item_feats):
        u = self.user_emb(user_idx)
        i = self.item_mlp(item_feats)

        u = F.normalize(u, dim=1)
        i = F.normalize(i, dim=1)

        logits = (u * i).sum(dim=1) * self.temperature
        return logits

    def user_vector(self, user_idx):
        u = self.user_emb(user_idx)
        return F.normalize(u, dim=1)

    def item_vectors(self, item_feats):
        i = self.item_mlp(item_feats)
        return F.normalize(i, dim=1)

### 3. Negative sampling

In [14]:
def sample_negatives_global(batch_size, n_items):
    return torch.randint(0, n_items, (batch_size,))

### 4. Тренування Two‑Tower

In [15]:
model_twotower = TwoTower(len(users), n_genres, d=64, temperature=10.0)
opt = torch.optim.Adam(model_twotower.parameters(), lr=1e-3)
loss_fn = nn.BCEWithLogitsLoss()

BATCH = 2048
EPOCHS = 5

for epoch in range(EPOCHS):
    perm = torch.randperm(len(pos_u))
    pos_users = pos_u[perm]
    pos_items = pos_i[perm]

    losses = []

    for start in range(0, len(pos_users), BATCH):
        end = start + BATCH
        bu = pos_users[start:end]
        bi = pos_items[start:end]

        # позитиви
        pos_feats = item_feats[bi]

        # негативи
        bn = sample_negatives_global(len(bu), M)
        neg_feats = item_feats[bn]

        # логіти
        pos_logits = model_twotower(bu, pos_feats)
        neg_logits = model_twotower(bu, neg_feats)

        # лейбли
        labels = torch.cat([
            torch.ones_like(pos_logits),
            torch.zeros_like(neg_logits)
        ])
        logits = torch.cat([pos_logits, neg_logits])

        loss = loss_fn(logits, labels)

        opt.zero_grad()
        loss.backward()
        opt.step()

        losses.append(loss.item())

    print(f"Epoch {epoch+1}: loss={sum(losses)/len(losses):.4f}")

Epoch 1: loss=0.8219
Epoch 2: loss=0.7786
Epoch 3: loss=0.7510
Epoch 4: loss=0.7306
Epoch 5: loss=0.7146


### 5. Precompute item vectors (FAISS‑style retrieval)

In [16]:
with torch.no_grad():
    item_vecs = model_twotower.item_vectors(item_feats)   # (M, d)

### 6. Recall@10 через Two‑Tower

In [17]:
def score_fn(user_idxs):
    with torch.no_grad():
        u_vecs = model_twotower.user_vector(user_idxs)        # (U, d)
        scores = u_vecs @ item_vecs.T                # (U, M)
    return scores

In [18]:
recall_twotower = recall_at_k(score_fn, k=10)
print("Recall@10 (TwoTower):", recall_twotower)

Recall@10 (TwoTower): 0.015044753380308513


### 7. Топ‑5 рекомендацій

In [19]:
def recommend(user_id, k=5):
    u_idx = user_to_idx[user_id]
    scores = score_fn(torch.tensor([u_idx]))[0].clone()

    # прибираємо вже бачене
    for i in seen_by_user[u_idx]:
        scores[i] = -1e9

    top = torch.topk(scores, k).indices.tolist()
    return [(title_of[items[i]], float(scores[i])) for i in top]

In [20]:
example_user = users[0]
recommend(example_user)

[('Midnight in the Garden of Good and Evil', 0.06500077992677689),
 ('The 19th Wife', 0.060294609516859055),
 ('Alias Grace', 0.05998872593045235),
 ('Shantaram', 0.05792900174856186),
 ('A Painted House', 0.054697342216968536)]

---
## Завдання 3. Concat-based ranking (NCF)

На відміну від Two-Tower (late fusion), тут **early fusion**: склеюємо ембединг користувача і ознаки книги в один вектор і пропускаємо через MLP, який сам моделює крос-взаємодії. Платою є те, що модель **не можна заіндексувати** — щоб знайти найкращу книгу, треба прогнати кожну пару (user, item). Тому її використовують лише на фінальному ранжуванні кількох кандидатів.

**Що зробити:**

1. Реалізуйте `NCF`: `concat(user_embedding, item_genre_features)` → MLP → один логіт.
2. Навчіть на тих самих позитивах/негативах.
3. Реалізуйте `rank_ncf(user_idx, candidate_idxs)` — ранжування заданого списку кандидатів за `sigmoid` логіта.


### 1. Архітектура NCF (early fusion)

user_emb(user_id) → u
item_genre_features → x
concat([u, x]) → MLP → логіт

### 2. Реалізація NCF

In [21]:
class NCF(nn.Module):
    def __init__(self, n_users, n_genres, d=64):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, d)

        self.mlp = nn.Sequential(
            nn.Linear(d + n_genres, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)   # один логіт
        )

    def forward(self, user_idx, item_feats):
        u = self.user_emb(user_idx)          # (B, d)
        x = item_feats                       # (B, n_genres)
        h = torch.cat([u, x], dim=1)         # (B, d+n_genres)
        logit = self.mlp(h).squeeze(1)       # (B,)
        return logit

### 3. Negative sampling (той самий, що в Two‑Tower)

In [22]:
def sample_negatives_global(batch_size, n_items):
    return torch.randint(0, n_items, (batch_size,))

### 4. Навчання NCF з BCEWithLogitsLoss

In [23]:
model_ncf = NCF(n_users=len(users), n_genres=n_genres, d=64)
opt = torch.optim.Adam(model_ncf.parameters(), lr=1e-3)
loss_fn = nn.BCEWithLogitsLoss()

BATCH = 2048
EPOCHS = 5

for epoch in range(EPOCHS):
    perm = torch.randperm(len(pos_u))
    pos_users = pos_u[perm]
    pos_items = pos_i[perm]

    losses = []

    for start in range(0, len(pos_users), BATCH):
        end = start + BATCH
        bu = pos_users[start:end]
        bi = pos_items[start:end]

        # позитиви
        pos_feats = item_feats[bi]

        # негативи
        bn = sample_negatives_global(len(bu), M)
        neg_feats = item_feats[bn]

        # логіти
        pos_logits = model_ncf(bu, pos_feats)
        neg_logits = model_ncf(bu, neg_feats)

        # лейбли
        labels = torch.cat([
            torch.ones_like(pos_logits),
            torch.zeros_like(neg_logits)
        ])
        logits = torch.cat([pos_logits, neg_logits])

        loss = loss_fn(logits, labels)

        opt.zero_grad()
        loss.backward()
        opt.step()

        losses.append(loss.item())

    print(f"Epoch {epoch+1}: loss={sum(losses)/len(losses):.4f}")

Epoch 1: loss=0.6888
Epoch 2: loss=0.6799
Epoch 3: loss=0.6754
Epoch 4: loss=0.6681
Epoch 5: loss=0.6584


### 5. Ранжування кандидатів: rank_ncf(user_idx, candidate_idxs)

У NCF ми не можемо попередньо обчислити item vectors — це early fusion, тому:
- для кожного кандидата треба прогнати forward
- це нормально, бо NCF використовують лише на фінальному ранжуванні 50–200 кандидатів

In [24]:
def rank_ncf(user_idx, candidate_idxs):
    u = torch.tensor([user_idx] * len(candidate_idxs))
    feats = item_feats[candidate_idxs]

    with torch.no_grad():
        logits = model_ncf(u, feats)
        scores = torch.sigmoid(logits)

    # сортуємо
    sorted_idx = torch.argsort(scores, descending=True)
    ranked_items = [candidate_idxs[i] for i in sorted_idx]

    return ranked_items, scores[sorted_idx]

### 6. Приклад: топ‑5 рекомендацій для користувача

In [25]:
def recommend_ncf(user_id, k=5):
    u_idx = user_to_idx[user_id]

    # кандидати = всі книги, крім тих, що вже бачив
    candidates = [i for i in range(M) if i not in seen_by_user[u_idx]]

    ranked, scores = rank_ncf(u_idx, candidates)

    top_items = ranked[:k]
    return [(title_of[items[i]], float(scores[j])) for j, i in enumerate(top_items)]

In [26]:
example_user = users[0]
recommend_ncf(example_user)

[("The Clan of the Cave Bear (Earth's Children, #1)", 0.768966019153595),
 ('Stardust', 0.768966019153595),
 ('Pride and Prejudice and Zombies (Pride and Prejudice and Zombies, #1)',
  0.7443512082099915),
 ('The Westing Game', 0.7391179203987122),
 ('The Book Thief', 0.7390737533569336)]

### 7. Recall@10 для NCF

In [27]:
def score_fn_ncf(user_idxs):
    with torch.no_grad():
        # user_idxs: tensor of shape (U,)
        U = len(user_idxs)
        # повторюємо кожного користувача для всіх item'ів
        user_rep = user_idxs.unsqueeze(1).repeat(1, M).reshape(-1)
        item_rep = torch.arange(M).repeat(U)

        feats = item_feats[item_rep]
        logits = model_ncf(user_rep, feats)
        scores = logits.reshape(U, M)
    return scores

In [28]:
recall_ncf = recall_at_k(score_fn_ncf, k=10)
print("Recall@10 (NCF):", recall_ncf)

Recall@10 (NCF): 0.024757189106836792


---
## Завдання 4. Двоетапний пайплайн Retrieval → Ranking

Поєднаємо все так, як це працює у великих системах: **Two-Tower швидко відбирає кандидатів** (retrieval серед усіх книг), а **NCF точно ранжує** цю коротку добірку.

**Що зробити:**

1. `retrieve(user_idx, n_candidates)` — топ-N книг за Two-Tower (Завдання 2), без уже побачених.
2. `recommend_pipeline(user_idx, n_candidates, top_k)` — прогнати кандидатів через `rank_ncf` (Завдання 3).
3. Показати для кількох користувачів: що відібрав retrieval і що залишив ranking.


### 1. Retrieval (Two‑Tower)
  
Two‑Tower повертає вектори користувача → множимо на item_vecs → беремо топ‑N.

In [29]:
def retrieve(user_idx, n_candidates=200):
    """
    Повертає top-N кандидатів за Two-Tower, без уже бачених книг.
    """
    with torch.no_grad():
        u_vec = model_twotower.user_vector(torch.tensor([user_idx]))  # (1, d)
        scores = (u_vec @ item_vecs.T).squeeze(0)                     # (M,)

    # прибираємо вже бачені
    s = scores.clone()
    for i in seen_by_user[user_idx]:
        s[i] = -1e9

    topN = torch.topk(s, n_candidates).indices.tolist()
    return topN

### 2. Ranking (NCF)

NCF ранжує тільки кандидатів, а не весь корпус.

In [30]:
def rank_ncf(user_idx, candidate_idxs):
    """
    Ранжує список кандидатів за NCF (sigmoid логіта).
    """
    u = torch.tensor([user_idx] * len(candidate_idxs))
    feats = item_feats[candidate_idxs]

    with torch.no_grad():
        logits = model_ncf(u, feats)
        scores = torch.sigmoid(logits)

    sorted_idx = torch.argsort(scores, descending=True)
    ranked_items = [candidate_idxs[i] for i in sorted_idx]

    return ranked_items, scores[sorted_idx]

### 3. Повний пайплайн Retrieval - Ranking

In [31]:
def recommend_pipeline(user_id, n_candidates=200, top_k=10):
    u_idx = user_to_idx[user_id]

    # 1) Retrieval
    candidates = retrieve(u_idx, n_candidates)

    # 2) Ranking only candidates
    ranked_items, scores = rank_ncf(u_idx, candidates)

    # 3) Top-k
    top_items = ranked_items[:top_k]
    top_scores = scores[:top_k]

    return [(title_of[items[i]], float(top_scores[j])) 
            for j, i in enumerate(top_items)]

### 4. Показати для кількох користувачів: що відібрав retrieval і що залишив ranking

In [32]:
def show_pipeline_example(user_id, n_candidates=20, top_k=5):
    u_idx = user_to_idx[user_id]

    print(f"\n=== USER {user_id} ===")

    # Retrieval
    candidates = retrieve(u_idx, n_candidates)
    print("\nRetrieval candidates (Two-Tower):")
    for i in candidates:
        print("  •", title_of[items[i]])

    # Ranking
    ranked, scores = rank_ncf(u_idx, candidates)
    print("\nRanking top-k (NCF):")
    for i in ranked[:top_k]:
        print("  →", title_of[items[i]])

In [33]:
for uid in users[:3]:
    show_pipeline_example(uid, n_candidates=30, top_k=5)


=== USER 9 ===

Retrieval candidates (Two-Tower):
  • Midnight in the Garden of Good and Evil
  • The 19th Wife
  • Alias Grace
  • Shantaram
  • A Painted House
  • The Godfather
  • The Hunt for Red October (Jack Ryan Universe, #4)
  • Kane and Abel (Kane and Abel, #1)
  • Death Comes to Pemberley
  • The Big Short: Inside the Doomsday Machine
  • Wolf Hall (Thomas Cromwell, #1)
  • Cutting for Stone
  • Little Bee
  • The Kitchen House
  • The Lowland
  • A Spool of Blue Thread
  • Stones from the River
  • And the Mountains Echoed
  • Snow Falling on Cedars
  • Eye of the Needle
  • Death on the Nile (Hercule Poirot, #17)
  • The Paying Guests
  • The Day of the Jackal
  • The Complete Sherlock Holmes
  • Murder at the Vicarage (Miss Marple, #1)
  • Murder on the Orient Express (Hercule Poirot, #10)
  • The Mysterious Affair at Styles (Hercule Poirot, #1)
  • The Murder of Roger Ackroyd (Hercule Poirot, #4)
  • The Complete Sherlock Holmes, Vol 2
  • A Study in Scarlet

Ranking to

**Питання:** навіщо ділити на два етапи, якщо можна ранжувати NCF одразу всі книги?

**Чому не ранжувати NCF одразу всі книги?**
Бо:
- повільно
- дорого
- не масштабується
- не працює на мільйонах item'ів
- не можна індексувати
- не можна робити ANN‑пошук

**Чому двоетапний пайплайн?**
Бо:
- Two‑Tower → швидко відбирає 100–500 кандидатів
- NCF → точно ранжує тільки їх
- працює в real‑time
- масштабується до мільйонів item'ів
- дає найкращу якість

---
## Завдання 5. Теоретичний блок (письмові відповіді)

Спираючись на лекцію та на те, що Ви щойно побачили на реальних даних, дайте розгорнуті відповіді в markdown-клітинці нижче.

1. **Чому Recall@10 такий низький?** На реальних даних усі моделі цього ДЗ дають скромний Recall@10. Назвіть щонайменше дві причини (підказки: бідні контентні ознаки — лише 12 жанрів; розрідженість; те, що val-лайки не охоплюють усіх книг, які користувач *міг би* вподобати).
2. **Як покращити якість, не змінюючи архітектуру?** Які додаткові ознаки книг і користувачів з Goodbooks можна було б під'єднати? (автор, рік, середній рейтинг, повний набір тегів через TF-IDF, текстові ембединги опису через BERT...)
3. **Diversity.** Якщо користувач любить фентезі, чому не варто показувати йому 10 фентезі-книг підряд? Як технічно підмішати різноманітність?
4. **Freshness / cold start.** Нова книга має 0 оцінок. Який підхід цього ДЗ зможе рекомендувати її одразу, а який — ні? Чому?
5. **Watch time > CTR (з лекції).** Поясніть, чому YouTube оптимізує час перегляду, а не CTR, і як це технічно вшито у weighted logistic regression.


### 1. Чому Recall@10 такий низький?
- бідні контентні ознаки — лише 12 жанрів; розрідженість; те, що val-лайки не охоплюють усіх книг, які користувач міг би вподобати
- профіль користувача усереднений, немає персональних ознак користувача
- жанрові ознаки надто грубі
- вектори книг дуже схожі
- Goodreads — шумний датасет

### 2. Як покращити якість, не змінюючи архітектуру?

Можна значно підняти якість, просто додавши кращі ознаки книг і користувачів.

Ознаки книг (items):
- Автор (one‑hot або embedding)
- Рік публікації
- Середній рейтинг
- Кількість оцінок
- Повний набір тегів (≈1000+) через TF‑IDF
- Опис книги → BERT / Sentence‑BERT embedding
- Жанрові пропорції, а не бінарні ознаки
- Серія (Harry Potter #1, #2, …)
- Видавництво

Ознаки користувачів (users):
- Середній рейтинг користувача
- Розподіл жанрів у його історії
- Embedding історії читання (mean pooling item embeddings)
- Часові патерни (коли читає)
- Схильність до високих/низьких оцінок

### 3. Diversity: чому не варто показувати 10 фентезі‑книг підряд?

Навіть якщо користувач любить фентезі, показувати 10 однакових книг — погана ідея:
- користувачеві швидко набридає
- рекомендації стають одноманітними
- знижується engagement
- користувач може пропустити інші цікаві жанри
- система стає негнучкою

Як технічно підмішати різноманітність?
- MMR (Maximal Marginal Relevance) score =𝜆 ⋅ relevance − (1 − 𝜆) ⋅ similarity_to_previous
- Penalize same‑genre repetition - якщо в топі вже є 3 фентезі → знижуємо скор наступних фентезі
- Re‑ranking з diversity penalty - після ranking додаємо diversity‑loss
- Clustering - беремо по 1–2 книги з кожного кластера embedding‑простору
- Stochastic sampling - замість топ‑10 беремо top‑50 і вибираємо 10 з випадковим шумом

### 4. Freshness / cold start:

Може рекомендувати одразу: Two‑Tower (content‑based). Бо Two‑Tower використовує жанрові ознаки книги.

Навіть якщо книга нова і не має жодного рейтингу:
- у неї є жанри
- item tower може побудувати embedding
- retrieval може знайти її як схожу на інші книги

Не може рекомендувати одразу: VSM та NCF (interaction‑based)
- VSM потребує, щоб хтось лайкнув книгу → інакше вона не має embedding
- NCF потребує позитивних/негативних прикладів → нова книга не з’явиться у train
- NCF не може бути індексований, тому нова книга не потрапить у ranking без retraining

### 5. Watch time > CTR

CTR (click‑through rate):
- легко накрутити клікбейтом
- не відображає реальної цінності контенту
- користувач може клікнути і закрити відео через 2 секунди

Watch time:
- вимірює реальну залученість
- корелює з задоволенням користувача
- краще прогнозує довгострокове утримання
- не стимулює клікбейт

YouTube використовує weighted logistic regression, де:
- позитивний приклад = відео, яке користувач дивився
- вага прикладу = watch time (в секундах або хвилинах)

Тобто:

$$loss = 𝑤_𝑖 ⋅ BCE(𝑦_𝑖,\hat{𝑦}_𝑖)$$

де 𝑤𝑖 = watch_time𝑖

Це означає:
- відео, яке дивилися 20 хвилин - має в 20 разів більший вплив
- відео, яке закрили через 5 секунд - майже не впливає
- модель вчиться передбачати довгий перегляд, а не просто клік